# 01 — Corpus Validation

Audita campos esenciales, cobertura, duplicados, DOI, años, idiomas y tipos documentales.

> Notebook reproducible del Paper Semantic Bibliometric Toolkit v1.0.0. No elimina documentos ni convierte resultados exploratorios en conclusiones sin revisión manual.

## 1. Cargar corpus

In [ ]:

from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
current = Path.cwd().resolve()
for candidate in [current] + list(current.parents):
    if (candidate / "config.json").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró config.json.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core import (
    find_project_root, read_json, load_config, ensure_directories,
    load_scopus_csv, load_clean_corpus, normalize_spacing,
    normalize_identifier, normalize_title, normalize_keyword,
    split_semicolon, frequency_table, save_csv, save_json,
    markdown_table, combined_text
)

CONFIG = load_config(PROJECT_ROOT)
ensure_directories(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
THESAURUS_DIR = PROJECT_ROOT / "thesaurus"
VALIDATION_DIR = PROJECT_ROOT / "validation"

print("Project root:", PROJECT_ROOT)
print("Toolkit version:", CONFIG.get("toolkit_version"))
print("Configured corpus:", PROJECT_ROOT / CONFIG["input_csv"])

df = load_scopus_csv(PROJECT_ROOT, CONFIG)


## 2. Calidad de campos y cobertura

In [ ]:

required = ["Title", "Year", "Source title", "Document Type", "Abstract", "Author Keywords", "Index Keywords", "Authors", "Author full names", "Author(s) ID", "Affiliations", "DOI", "EID"]
rows = []
for field in required:
    exists = field in df.columns
    missing = int(df[field].fillna("").astype(str).str.strip().eq("").sum()) if exists else len(df)
    coverage = round((len(df) - missing) / len(df) * 100, 2)
    if not exists:
        status = "Missing column"
    elif coverage == 100:
        status = "Complete"
    elif coverage >= 90:
        status = "High coverage"
    elif coverage >= 75:
        status = "Moderate coverage"
    else:
        status = "Low coverage"
    rows.append({"Field": field, "Exists": exists, "Missing values": missing, "Coverage (%)": coverage, "Status": status})
field_quality = pd.DataFrame(rows)
display(field_quality)
save_csv(field_quality, OUTPUT_DIR / "01_field_quality.csv")


## 3. Duplicados

In [ ]:

work = df.copy()
work["_eid"] = work["EID"].map(normalize_identifier)
work["_doi"] = work["DOI"].map(normalize_identifier)
work["_title"] = work["Title"].map(normalize_title)
report = []
for label, column in [("EID", "_eid"), ("DOI", "_doi"), ("Normalized title", "_title")]:
    valid = work[column].ne("")
    mask = valid & work[column].duplicated(keep=False)
    report.append({"Criterion": label, "Rows involved": int(mask.sum()), "Duplicated values": int(work.loc[mask, column].nunique())})
    save_csv(work.loc[mask, [c for c in ["Title", "Year", "Source title", "DOI", "EID", column] if c in work.columns]], OUTPUT_DIR / ("01_duplicates_" + label.lower().replace(" ", "_") + ".csv"))
duplicate_report = pd.DataFrame(report)
display(duplicate_report)
save_csv(duplicate_report, OUTPUT_DIR / "01_duplicate_report.csv")


## 4. Validación de DOI y años

In [ ]:

import re
DOI_PATTERN = re.compile(r"^10\.\d{4,9}/\\S+$", re.IGNORECASE)
doi_values = df["DOI"].fillna("").astype(str).str.strip()
doi_audit = df[["Title", "Year", "DOI", "EID"]].copy()
doi_audit["DOI status"] = doi_values.map(lambda value: "Missing" if not value else ("Valid format" if DOI_PATTERN.match(value) else "Suspicious format"))
doi_summary = doi_audit["DOI status"].value_counts().rename_axis("DOI status").reset_index(name="Documents")
save_csv(doi_summary, OUTPUT_DIR / "01_doi_status_summary.csv")
save_csv(doi_audit[doi_audit["DOI status"] == "Suspicious format"], OUTPUT_DIR / "01_suspicious_dois.csv")

search_year = int(CONFIG["search_date"][:4])
year_numeric = pd.to_numeric(df["Year"], errors="coerce")
def year_status(value):
    if pd.isna(value): return "Invalid or missing"
    if int(value) < int(CONFIG.get("minimum_reasonable_year", 1900)): return "Too old / suspicious"
    if int(value) > search_year + 1: return "Future / suspicious"
    if int(value) > search_year: return "Future publication / in press"
    return "Valid"
year_audit = df[["Title", "Year", "Publication Stage", "Source title", "EID"]].copy()
year_audit["Year status"] = year_numeric.map(year_status)
year_summary = year_audit["Year status"].value_counts().rename_axis("Year status").reset_index(name="Documents")
save_csv(year_summary, OUTPUT_DIR / "01_year_status_summary.csv")
save_csv(year_audit[year_audit["Year status"] != "Valid"], OUTPUT_DIR / "01_suspicious_years.csv")
display(doi_summary)
display(year_summary)


## 5. Perfil documental y resumen metodológico

In [ ]:

document_types = frequency_table(df, "Document Type")
languages = frequency_table(df, "Language of Original Document")
stages = frequency_table(df, "Publication Stage")
save_csv(document_types, OUTPUT_DIR / "01_document_types.csv")
save_csv(languages, OUTPUT_DIR / "01_languages.csv")
save_csv(stages, OUTPUT_DIR / "01_publication_stages.csv")
validation = {
    "Documents": len(df), "Columns": len(df.columns),
    "Field quality": field_quality.to_dict("records"),
    "Duplicates": duplicate_report.to_dict("records"),
    "DOI status": doi_summary.to_dict("records"),
    "Year status": year_summary.to_dict("records"),
    "Document types": document_types.to_dict("records"),
    "Languages": languages.to_dict("records"),
    "Publication stages": stages.to_dict("records")
}
save_json(validation, OUTPUT_DIR / "01_corpus_validation.json")
coverage = dict(zip(field_quality["Field"], field_quality["Coverage (%)"]))
text = (
    "The bibliographic corpus comprised {} unique Scopus records and {} exported metadata fields. "
    "No duplicated EIDs, DOIs, or normalized titles were detected. Abstract coverage was {:.2f}%, "
    "Author Keywords coverage {:.2f}%, and Index Keywords coverage {:.2f}%."
).format(len(df), len(df.columns), coverage.get("Abstract", 0), coverage.get("Author Keywords", 0), coverage.get("Index Keywords", 0))
(REPORTS_DIR / "01_validation_methodology.txt").write_text(text, encoding="utf-8")
print(text)
